# Policy Rule Training Pipeline

This notebook combines dataset generation and model training for fine-tuning Qwen2.5-1.5B on Rego policy rules.

## Steps:
1. **Setup & Configuration** - Set paths and parameters
2. **Generate Dataset** - Parse Rego files and create training examples
3. **Validate Dataset** - Check dataset quality and statistics
4. **Prepare Training** - Load and tokenize data
5. **Train Model** - Fine-tune with LoRA
6. **Evaluate** - Check training results


## 0. Clone Repository

First, clone the repository if it doesn't exist locally.


In [ ]:
import subprocess
import os
from pathlib import Path

# Repository configuration
REPO_URL = "git@github.com:joejstuart/policy_training.git"
REPO_NAME = "policy_training"

# Determine where to clone (current directory or parent)
WORK_DIR = Path.cwd()

# Check if we're already in the repo (look for policy directory)
if (WORK_DIR / "policy").exists():
    # Already in the repo root
    REPO_ROOT = WORK_DIR
    print(f"Already in repository at: {REPO_ROOT}")
elif WORK_DIR.name == "qwen2.5_model" and (WORK_DIR.parent / "policy").exists():
    # In qwen2.5_model subdirectory, parent is repo root
    REPO_ROOT = WORK_DIR.parent
    print(f"Using parent directory as repository root: {REPO_ROOT}")
elif WORK_DIR.name == "qwen2.5_model":
    # In qwen2.5_model but not in repo, clone to parent
    REPO_ROOT = WORK_DIR.parent / REPO_NAME
    print(f"Repository will be cloned to: {REPO_ROOT}")
else:
    # Running from elsewhere, clone here
    REPO_ROOT = WORK_DIR / REPO_NAME
    print(f"Repository will be cloned to: {REPO_ROOT}")

# Clone or update repository (only if not already in repo)
if (REPO_ROOT / "policy").exists():
    print(f"✓ Repository found at {REPO_ROOT}")
    print("Updating repository...")
    try:
        result = subprocess.run(
            ["git", "pull"],
            cwd=REPO_ROOT,
            check=True,
            capture_output=True,
            text=True
        )
        print("✓ Repository updated")
    except subprocess.CalledProcessError as e:
        print(f"⚠ Warning: Could not update repository: {e}")
        print("  Continuing with existing code...")
elif REPO_ROOT.exists():
    print(f"⚠ Directory exists but doesn't look like a repository: {REPO_ROOT}")
    print("  Attempting to clone anyway...")
    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_ROOT)],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"✓ Repository cloned to {REPO_ROOT}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error cloning repository: {e}")
        print(f"  Make sure you have SSH access to {REPO_URL}")
        print(f"  Or clone manually: git clone {REPO_URL} {REPO_ROOT}")
        raise
else:
    print(f"Cloning repository from {REPO_URL}...")
    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_ROOT)],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"✓ Repository cloned to {REPO_ROOT}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error cloning repository: {e}")
        print(f"  Make sure you have SSH access to {REPO_URL}")
        print(f"  Or clone manually: git clone {REPO_URL} {REPO_ROOT}")
        raise

# Verify repository structure
if not (REPO_ROOT / "policy").exists():
    print(f"⚠ Warning: 'policy' directory not found in {REPO_ROOT}")
    print("  Repository may not have been cloned correctly")
    raise FileNotFoundError(f"Repository structure invalid: {REPO_ROOT}")
else:
    print(f"✓ Repository structure verified")
    
# Make REPO_ROOT available globally for subsequent cells
print(f"\nRepository ready at: {REPO_ROOT}")


## 1. Setup & Configuration


In [ ]:
import json
import os
import sys
import re
import subprocess
import tempfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from collections import defaultdict
import random
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType

# Set tokenizers parallelism to avoid warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Add cloned repository to Python path
# REPO_ROOT is set in the previous cell (from cloning)
sys.path.insert(0, str(REPO_ROOT))
if (REPO_ROOT / "qwen2.5_model").exists():
    sys.path.insert(0, str(REPO_ROOT / "qwen2.5_model"))

print("✓ Imports loaded and paths configured")


In [ ]:
# Configuration
# REPO_ROOT is set in the clone cell above

POLICY_RELEASE_DIR = REPO_ROOT / "policy" / "release"
POLICY_LIB_DIR = REPO_ROOT / "policy" / "lib"
RELEASE_LIB_DIR = REPO_ROOT / "policy" / "release" / "lib"

# Dataset paths (save in notebook directory, not repo)
NOTEBOOK_DIR = Path.cwd()
TRAIN_PATH = NOTEBOOK_DIR / "train.jsonl"
EVAL_PATH = NOTEBOOK_DIR / "eval.jsonl"
DATASET_SUMMARY_PATH = NOTEBOOK_DIR / "dataset_summary.json"

# Training configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = NOTEBOOK_DIR / "qwen2.5-rego-policy-lora"
MAX_SEQ_LEN = 1024
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 5e-5
NUM_EPOCHS = 3
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Dataset generation settings
TRAIN_SPLIT = 0.9  # 90% train, 10% eval
MAX_TOKENS = 1024

print(f"Repository root: {REPO_ROOT}")
print(f"Policy release dir: {POLICY_RELEASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Dataset files will be saved to: {NOTEBOOK_DIR}")


## 2. Generate Dataset

Import the dataset generation functions and run them.


In [ ]:
# Import dataset generation functions from cloned repository
try:
    from qwen2.5_model.generate_dataset import (
        parse_rego_file,
        generate_implement_example,
        generate_refactor_example,
        validate_rego_code,
        example_to_jsonl,
        RuleExample,
        RegoFile,
        extract_used_imports,
        extract_used_helpers,
        build_context
    )
    print("✓ Dataset generation functions imported from repository")
except ImportError:
    # Fallback: try direct import if running from repo
    try:
        from generate_dataset import (
            parse_rego_file,
            generate_implement_example,
            generate_refactor_example,
            validate_rego_code,
            example_to_jsonl,
            RuleExample,
            RegoFile,
            extract_used_imports,
            extract_used_helpers,
            build_context
        )
        print("✓ Dataset generation functions imported (direct)")
    except ImportError as e:
        print(f"❌ Could not import from generate_dataset.py: {e}")
        print(f"  Make sure the repository is cloned at: {REPO_ROOT}")
        print(f"  And that qwen2.5_model/generate_dataset.py exists")
        raise

# Helper functions to match the notebook's expected interface
def parse_rego_files(policy_dir):
    """Parse all Rego files in a directory."""
    rego_files = []
    for rego_file in policy_dir.rglob("*.rego"):
        if "_test.rego" in rego_file.name:
            continue
        parsed = parse_rego_file(rego_file)
        if parsed and parsed.rules:
            rego_files.append(parsed)
    return rego_files

def generate_training_examples(rego_file, lib_dir, release_lib_dir):
    """Generate training examples from a parsed Rego file."""
    examples = []
    for rule in rego_file.rules:
        # Generate implement example
        impl_example = generate_implement_example(rego_file, rule, "")
        if impl_example:
            examples.append(impl_example)
        
        # Generate refactor example (60% chance)
        if random.random() < 0.6:
            refactor_example = generate_refactor_example(rego_file, rule, "")
            if refactor_example:
                examples.append(refactor_example)
    return examples

def split_train_eval(examples, train_split=0.9):
    """Split examples into train and eval sets."""
    random.shuffle(examples)
    
    # Separate by task type for better distribution
    implement_examples = [e for e in examples if e.task_type == "implement"]
    refactor_examples = [e for e in examples if e.task_type == "refactor"]
    
    # Split each type
    impl_split = int(len(implement_examples) * train_split)
    ref_split = int(len(refactor_examples) * train_split)
    
    train_examples = implement_examples[:impl_split] + refactor_examples[:ref_split]
    eval_examples = implement_examples[impl_split:] + refactor_examples[ref_split:]
    
    # Ensure eval has both types
    if not any(e.task_type == "implement" for e in eval_examples) and implement_examples:
        if impl_split > 0:
            train_examples.remove(implement_examples[impl_split-1])
            eval_examples.append(implement_examples[impl_split-1])
    
    if not any(e.task_type == "refactor" for e in eval_examples) and refactor_examples:
        if ref_split > 0:
            train_examples.remove(refactor_examples[ref_split-1])
            eval_examples.append(refactor_examples[ref_split-1])
    
    return train_examples, eval_examples

def write_jsonl(examples, file_path):
    """Write examples to JSONL file."""
    with open(file_path, "w", encoding="utf-8") as f:
        for example in examples:
            f.write(example_to_jsonl(example) + "\n")

print("✓ Helper wrapper functions defined")


In [ ]:
# Parse all Rego files
print("Parsing Rego files...")
rego_files = parse_rego_files(POLICY_RELEASE_DIR)
print(f"✓ Parsed {len(rego_files)} Rego files")

# Show some statistics
total_rules = sum(len(f.rules) for f in rego_files)
print(f"  Total rules found: {total_rules}")

# Show packages
packages = set(f.package for f in rego_files)
print(f"  Packages: {len(packages)}")
print(f"  Sample packages: {list(packages)[:5]}")


In [ ]:
# Generate training examples
print("Generating training examples...")
examples = []

for rego_file in rego_files:
    file_examples = generate_training_examples(rego_file, POLICY_LIB_DIR, RELEASE_LIB_DIR)
    examples.extend(file_examples)
    if len(examples) % 50 == 0:
        print(f"  Generated {len(examples)} examples...")

print(f"✓ Generated {len(examples)} total examples")

# Count by task type
task_types = defaultdict(int)
for ex in examples:
    task_types[ex.task_type] += 1
print(f"  Task types: {dict(task_types)}")


In [ ]:
# Validate examples
print("Validating examples...")
valid_examples = []
invalid_count = 0

for i, example in enumerate(examples):
    if i % 50 == 0:
        print(f"  Validated {i}/{len(examples)}...")
    
    # Get package and imports from source file (more reliable than parsing context)
    # source_file might be relative, try both repo root and absolute
    source_path = Path(example.source_file)
    if not source_path.is_absolute():
        # Try relative to repo root
        source_path = REPO_ROOT / example.source_file
    if not source_path.exists():
        # Try as absolute path
        source_path = Path(example.source_file)
    
    if source_path.exists():
        parsed_source = parse_rego_file(source_path)
        package = parsed_source.package if parsed_source else ""
        imports = parsed_source.imports if parsed_source else []
    else:
        # Fallback: extract from context
        package = ""
        if example.context:
            match = re.search(r'package\s+(\S+)', example.context)
            if match:
                package = match.group(1)
        
        imports = []
        if example.context:
            import_matches = re.findall(r'import\s+([^\n]+)', example.context)
            imports = [imp.strip() for imp in import_matches]
    
    # Validate output code
    is_valid, formatted_code, error_msg = validate_rego_code(
        example.output_code,
        package=package,
        imports=imports
    )
    
    if is_valid:
        # Update with formatted code
        example.output_code = formatted_code
        valid_examples.append(example)
    else:
        invalid_count += 1
        if invalid_count <= 5:  # Show first 5 errors
            print(f"    Invalid example: {error_msg[:100]}")

print(f"✓ Validated: {len(valid_examples)} valid, {invalid_count} invalid")


In [ ]:
# Split into train/eval
train_examples, eval_examples = split_train_eval(valid_examples, TRAIN_SPLIT)
print(f"✓ Split: {len(train_examples)} train, {len(eval_examples)} eval")

# Write to JSONL files
write_jsonl(train_examples, TRAIN_PATH)
write_jsonl(eval_examples, EVAL_PATH)
print(f"✓ Wrote {TRAIN_PATH}")
print(f"✓ Wrote {EVAL_PATH}")

# Create summary
summary = {
    "total_examples": len(valid_examples),
    "train_examples": len(train_examples),
    "eval_examples": len(eval_examples),
    "task_types": dict(task_types),
    "invalid_count": invalid_count
}
with open(DATASET_SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Wrote {DATASET_SUMMARY_PATH}")


## 3. Validate Dataset

Check dataset statistics and sample examples.


In [ ]:
# Load and display summary
with open(DATASET_SUMMARY_PATH) as f:
    summary = json.load(f)

print("Dataset Summary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Sample a few examples
print("\nSample Training Examples:\n")
with open(TRAIN_PATH) as f:
    for i, line in enumerate(f):
        if i >= 3:  # Show first 3
            break
        example = json.loads(line)
        print(f"Example {i+1} ({example['task_type']}):")
        print(f"  Instruction: {example['instruction'][:100]}...")
        print(f"  Context length: {len(example.get('context', ''))} chars")
        print(f"  Output code length: {len(example['output_code'])} chars")
        print()


## 4. Prepare Training

Load tokenizer, create dataset class, and prepare data loaders.


In [ ]:
# Load tokenizer
print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded (vocab size: {len(tokenizer)})")


In [ ]:
# System prompt
QWEN_SYSTEM_PROMPT = (
    "You are an expert Rego/OPA policy assistant. "
    "You follow instructions carefully and emit valid Rego code using "
    "Conforma's preferred patterns (deny contains result, METADATA, result_helper, etc). "
    "Only use helpers that are provided in the context - never invent new helper functions."
)

def build_messages_from_example(example):
    """Build chat messages from policy training example."""
    messages = [
        {"role": "system", "content": QWEN_SYSTEM_PROMPT}
    ]
    
    # Build user message
    user_parts = []
    
    if "context" in example:
        user_parts.append(example["context"])
    
    if "instruction" in example:
        user_parts.append("\n" + example["instruction"])
    
    if example.get("task_type") == "refactor" and "input_code" in example:
        user_parts.append("\n\nCode to refactor:\n```rego\n" + example["input_code"] + "\n```")
    
    user_content = "\n".join(user_parts)
    messages.append({"role": "user", "content": user_content})
    
    if "output_code" in example:
        messages.append({"role": "assistant", "content": example["output_code"]})
    
    return messages


In [ ]:
# Dataset class
class PolicyDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        
        # Load examples
        jsonl_path = Path(jsonl_path)
        if not jsonl_path.exists():
            raise FileNotFoundError(f"Dataset file not found: {jsonl_path}")
        
        with open(jsonl_path) as f:
            for line in f:
                if line.strip():
                    self.examples.append(json.loads(line))
        
        # Pre-tokenize all examples
        print(f"Pre-tokenizing {len(self.examples)} examples...")
        self.tokenized = []
        for i, example in enumerate(self.examples):
            if i % 50 == 0:
                print(f"  Tokenized {i}/{len(self.examples)}...")
            
            messages = build_messages_from_example(example)
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            
            encoded = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding=False
            )
            
            self.tokenized.append({
                "input_ids": encoded["input_ids"],
                "attention_mask": encoded["attention_mask"]
            })
        
        print(f"✓ Pre-tokenization complete")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.tokenized[idx]

print("✓ Dataset class defined")


In [ ]:
# Create datasets
print("Creating training dataset...")
train_dataset = PolicyDataset(TRAIN_PATH, tokenizer, max_length=MAX_SEQ_LEN)

print("\nCreating eval dataset...")
eval_dataset = PolicyDataset(EVAL_PATH, tokenizer, max_length=MAX_SEQ_LEN)

print(f"\n✓ Datasets ready:")
print(f"  Train: {len(train_dataset)} examples")
print(f"  Eval: {len(eval_dataset)} examples")


## 5. Train Model

Load base model, configure LoRA, and start training.


In [ ]:
# Load base model
print(f"Loading base model: {MODEL_NAME}...")

# Detect device
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

dtype = torch.bfloat16 if device != "cpu" else torch.float32

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map={"": device} if device != "cpu" else None,
    trust_remote_code=True
)

if device == "cpu":
    base_model = base_model.to(device)

print(f"✓ Base model loaded")


In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print(f"✓ LoRA configured and applied")


In [ ]:
# Training arguments
# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,
    logging_steps=10,
    eval_steps=50,
    save_steps=100,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    fp16=device != "cpu" and device != "mps",  # fp16 for CUDA
    bf16=device == "mps",  # bf16 for MPS (Apple Silicon)
    report_to="none",
    remove_unused_columns=False
)

print("✓ Training arguments configured")


In [ ]:
# Data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("✓ Data collator created")


In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✓ Trainer created")


In [ ]:
# Start training
print("Starting training...")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Training for {NUM_EPOCHS} epochs")
print(f"Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print()

trainer.train()

print("\n✓ Training complete!")


In [ ]:
# Save final model
print(f"Saving model to {OUTPUT_DIR}...")
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to {OUTPUT_DIR}")


## 6. Evaluate

Check training metrics and sample outputs.


In [ ]:
# Load training history
checkpoints = list(OUTPUT_DIR.glob("checkpoint-*"))

if checkpoints:
    try:
        latest_checkpoint = max(checkpoints, key=lambda p: int(p.name.split("-")[1]))
        trainer_state_path = latest_checkpoint / "trainer_state.json"
        
        if trainer_state_path.exists():
            with open(trainer_state_path) as f:
                state = json.load(f)
            
            print("Training History (last 10 entries):")
            if "log_history" in state:
                for entry in state["log_history"][-10:]:
                    if "loss" in entry:
                        step = entry.get("step", "?")
                        loss = entry.get("loss", "?")
                        eval_loss = entry.get("eval_loss", "?")
                        print(f"  Step {step}: loss={loss:.4f}, eval_loss={eval_loss:.4f}")
        else:
            print("No trainer_state.json found in checkpoints")
    except (ValueError, KeyError) as e:
        print(f"Could not parse checkpoint names: {e}")
        print(f"Found {len(checkpoints)} checkpoints")
else:
    print("No checkpoints found")


In [ ]:
# Test inference on a sample
print("\nTesting inference on a sample example...")

if not EVAL_PATH.exists():
    print(f"⚠ Eval file not found: {EVAL_PATH}")
    print("  Skipping inference test")
else:
    with open(EVAL_PATH) as f:
        first_line = f.readline().strip()
        if not first_line:
            print("⚠ Eval file is empty")
        else:
            sample = json.loads(first_line)
            
            messages = build_messages_from_example(sample)
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            
            inputs = tokenizer(text, return_tensors="pt").to(device)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.7,
                    do_sample=True
                )
            
            generated = tokenizer.decode(outputs[0], skip_special_tokens=False)
            assistant_text = tokenizer.apply_chat_template(
                messages + [{"role": "assistant", "content": ""}],
                tokenize=False,
                add_generation_prompt=True
            )
            
            if generated.startswith(assistant_text):
                response = generated[len(assistant_text):].strip()
            else:
                # Try to extract just the assistant response
                response = generated.split("assistant\n")[-1].strip() if "assistant\n" in generated else generated
            
            print("\nSample Input:")
            print(sample.get("instruction", "")[:200])
            print("\nGenerated Output:")
            print(response[:500])
            print("\nExpected Output:")
            print(sample.get("output_code", "")[:500])
